
<div class="alert alert-block alert-info">  
    
## Import & Connections
</div>

In [1]:
import numpy as np
import pandas as pd
import hashlib
import glob
import time
import os
import re
import matplotlib.pyplot as plt

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 50)

In [3]:
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [10]:
start_date = '20250115'
end_date = '20250122'
response_type = 'ONLINE_SALES'
pagename = 'PostOrderScreen'
campaign_name = 'Exp1_MaxBanner_Zepto'
format = 'maxBanner'

In [5]:
print(os.getcwd())
local_dataset = '/Users/E2074/local-datasets/customer/ads-monetisation/page-format-experiment/max-banner'
print(local_dataset)

/Users/E2074/analytics/customer/Ads-Monetisation/page-format-experiment
/Users/E2074/local-datasets/customer/ads-monetisation/page-format-experiment/max-banner


<div class="alert alert-block alert-warning">  
    
## Datasets & Functions
</div>

In [6]:
start = datetime.strptime(start_date, '%Y%m%d')
end = datetime.strptime(end_date, '%Y%m%d')

# Generate the list of dates
date_range = [(start + timedelta(days=i)).strftime('%Y%m%d') for i in range((end - start).days + 1)]
print(date_range)

['20250115', '20250116', '20250117', '20250118', '20250119', '20250120', '20250121', '20250122']


In [14]:
def get_customer_metadata(date_range):

    all_df = []
    
    for date in date_range:
    
        customer_metadata = f"""
        
            with response as (
            
            select
                distinct
                yyyymmdd,
                json_extract_scalar(metadata1, '$.campaignName') AS campaignName,
                json_extract_scalar(metadata1, '$.slotUnitId') AS slotUnitId,
                city,
                os,
                userid,
                service,
                orderstatus,
                -- totalcardcount,
                -- cardposition,
                coalesce(aduuid, adid) response_id
            from 
                canonical.iceberg_log_telemetry_ads_impressions_immutable
            where 
                yyyymmdd = '{date}'
                and os = 'android'
                and responsetype = '{response_type}'
                and eventname = 'Ad_Response'
                and pagename = '{pagename}'
                and format = '{format}'
                and regexp_like(metadata1, '{campaign_name}')
            ),
            
            impression as (
            
            select
                distinct
                yyyymmdd,
                userid,
                coalesce(aduuid, adid) impression_id
            from 
                canonical.iceberg_log_telemetry_ads_impressions_immutable
            where 
                yyyymmdd = '{date}'
                and os = 'android'
                and responsetype = '{response_type}'
                and eventname = 'Ad_Impression'
                and pagename = '{pagename}'
                and format = '{format}'
                and regexp_like(metadata1, '{campaign_name}')
            ),
            
            viewable_impression as (
            
            select
                distinct
                yyyymmdd,
                userid,
                coalesce(aduuid, adid) viewable_impression_id
            from 
                canonical.iceberg_log_telemetry_ads_impressions_immutable
            where 
                yyyymmdd = '{date}'
                and os = 'android'
                and responsetype = '{response_type}'
                and eventname = 'Ad_Viewable_Impression'
                and pagename = '{pagename}'
                and format = '{format}'
                and regexp_like(metadata1, '{campaign_name}')
            ),
            
            click as (
            
            select
                distinct
                yyyymmdd,
                userid,
                coalesce(aduuid, adid) click_id
            from 
                canonical.iceberg_log_telemetry_ads_impressions_immutable
            where 
                yyyymmdd = '{date}'
                and os = 'android'
                and responsetype = '{response_type}'
                and eventname = 'Ad_Click'
                and pagename = '{pagename}'
                and format = '{format}'
            )
            
            select
                a.*,
                impression_id,
                coalesce(click_id, viewable_impression_id) viewable_impression_id,
                click_id
            from 
                response a
            left join 
                impression b
                on a.yyyymmdd = b.yyyymmdd
                and a.userid = b.userid
                and a.response_id = b.impression_id
            left join 
                viewable_impression c
                on b.yyyymmdd = c.yyyymmdd
                and b.userid = c.userid
                and b.impression_id = c.viewable_impression_id
            left join 
                click d
                on b.yyyymmdd = d.yyyymmdd
                and b.userid = d.userid
                and b.impression_id = d.click_id
        """
        
        df = pd.read_sql(customer_metadata, connection)
        all_df.append(df)
        df.to_parquet(local_dataset + '/data_dump/Exp1_MaxBanner_Zepto_{}.parquet'.format(date), index=False)
    
    # Concatenate all DataFrames
    final_df = pd.concat(all_df, ignore_index=True)

    return final_df


# df_data = get_customer_metadata(date_range)

In [15]:
def get_customer_segment():    
    
    customer_segment = f"""
    
        with response as (

            select
                distinct
                userid
            from 
                canonical.iceberg_log_telemetry_ads_impressions_immutable
            where 
                yyyymmdd >= '{start_date}'
                and yyyymmdd <= '{end_date}'
                and os = 'android'
                and responsetype = '{response_type}'
                and eventname = 'Ad_Response'
                and pagename = '{pagename}'
                and format = '{format}'
                and regexp_like(metadata1, '{campaign_name}')
            ),

            segment as (
    
            select
                customer_id,
                gender_tag,
                taxi_ltr_segment,
                taxi_retention_segments,
                taxi_regularity_segment,
                taxi_fe_regularity_segment,
                taxi_income_segment,
                taxi_service_affinity,
                taxi_gc_tag
            from 
                datasets.iallocator_customer_segments
            where 
                run_date = '2025-01-14'
            )
            
            select
                userid,
                coalesce(gender_tag, 'UNKNOWN') gender,
                coalesce(taxi_ltr_segment, 'NEW') ltr_segment,
                coalesce(taxi_retention_segments, 'UNKNOWN') retention_segments,
                coalesce(taxi_regularity_segment, 'UNKNOWN') regularity_segment,
                coalesce(taxi_fe_regularity_segment, 'UNKNOWN') fe_regularity_segment,
                coalesce(taxi_income_segment, 'UNKNOWN') income_segment,
                coalesce(taxi_service_affinity, 'UNKNOWN') service_affinity,
                coalesce(taxi_gc_tag, 'UNKNOWN') gc_tag
            from 
                response
            left join 
                segment
                on userid = customer_id
            
    """
    
    df = pd.read_sql(customer_segment, connection)
    df.to_parquet(local_dataset + '/customer_segment.parquet', index=False)

    return df

# df_data = get_customer_segment()


<div class="alert alert-block alert-info">  
    
## Load data
</div>

In [16]:
def get_customer_metadata():
    
    # Use glob to find all Parquet files in the directory
    parquet_files = glob.glob(os.path.join(local_dataset + '/data_dump/', "*.parquet"))
    customer_metadata = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)

    customer_segment = pd.read_parquet(local_dataset + '/customer_segment.parquet')
    
    return customer_metadata, customer_segment

df_cux_metadata, df_cux_segment = get_customer_metadata()

In [17]:
df_cux_metadata.shape, df_cux_segment.shape

((1552970, 12), (579970, 9))

In [18]:
df_cux_metadata.head(1)

,yyyymmdd,campaignName,slotUnitId,city,os,userid,service,orderstatus,response_id,impression_id,viewable_impression_id,click_id
0,20250115,Exp1_MaxBanner_Zepto,PostOrderArrived3,Bangalore,android,5e5f1abf32e3140a0beaf16d,Auto,arrived,b57dbc1c-b86e-4519-a460-30a5478d9cd65e5f1abf32...,b57dbc1c-b86e-4519-a460-30a5478d9cd65e5f1abf32...,b57dbc1c-b86e-4519-a460-30a5478d9cd65e5f1abf32...,None


In [19]:
df_cux_segment.head(1)

,userid,gender,ltr_segment,retention_segments,regularity_segment,fe_regularity_segment,income_segment,service_affinity,gc_tag
0,65b0be3b38890ce05f64ee0f,MALE,PHH,INACTIVE,RARE_NEED,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN


In [20]:
df = pd.merge(df_cux_metadata, df_cux_segment, how='left', on = 'userid')
df.head(1).T

,0
yyyymmdd,20250115
campaignName,Exp1_MaxBanner_Zepto
slotUnitId,PostOrderArrived3
city,Bangalore
os,android
userid,5e5f1abf32e3140a0beaf16d
service,Auto
orderstatus,arrived
response_id,b57dbc1c-b86e-4519-a460-30a5478d9cd65e5f1abf32...
impression_id,b57dbc1c-b86e-4519-a460-30a5478d9cd65e5f1abf32...


<div class="alert alert-block alert-warning">  
    
## Funnel Func
</div>

In [21]:

def get_funnel(df, groupby):

    df_temp = df\
                .groupby(groupby)\
                .agg(responses = ('response_id', 'nunique'),
                     impression = ('impression_id', 'nunique'),
                     viewable_impression = ('viewable_impression_id', 'nunique'),
                     click = ('click_id', 'nunique'),
                    )\
                .reset_index()

    df_temp['% render rate'] = (df_temp['impression']*100.00/df_temp['responses']).round(2)
    df_temp['% view'] = (df_temp['viewable_impression']*100.00/df_temp['impression']).round(2)
    df_temp['% ctr'] = (df_temp['click']*100.00/df_temp['viewable_impression']).round(2)

    df_temp['format'] = format
    
    columns = ['format'] + groupby + ['responses', '% render rate', '% view', '% ctr']
    
    return df_temp[columns].sort_values(by='% ctr', ascending=False)

<div class="alert alert-block alert-success">  
    
## Analysis
</div>

In [22]:
df.head(5)

,yyyymmdd,campaignName,slotUnitId,city,os,userid,service,orderstatus,response_id,impression_id,viewable_impression_id,click_id,gender,ltr_segment,retention_segments,regularity_segment,fe_regularity_segment,income_segment,service_affinity,gc_tag
0,20250115,Exp1_MaxBanner_Zepto,PostOrderArrived3,Bangalore,android,5e5f1abf32e3140a0beaf16d,Auto,arrived,b57dbc1c-b86e-4519-a460-30a5478d9cd65e5f1abf32...,b57dbc1c-b86e-4519-a460-30a5478d9cd65e5f1abf32...,b57dbc1c-b86e-4519-a460-30a5478d9cd65e5f1abf32...,None,FEMALE,PHH,PLATINUM,DAILY,DAILY,MEDIUM_INCOME,ONLY_AUTO,UNKNOWN
1,20250115,Exp1_MaxBanner_Zepto,PostOrderOnTheWay3,Bangalore,android,676c0d81a353a7da26899fd1,Auto,onTheWay,b8e7c852-d0f3-488f-af24-f3f415854597676c0d81a3...,b8e7c852-d0f3-488f-af24-f3f415854597676c0d81a3...,None,None,MALE,PHH,GOLD,BI_WEEKLY,MONTHLY,UNKNOWN,ONLY_LINK,UNKNOWN
2,20250115,Exp1_MaxBanner_Zepto,PostOrderOnTheWay3,Kolkata,android,5d0a2fae1c64cc470f63ece1,Link,onTheWay,b07442b2-3166-4233-b884-90f6803d1bfd5d0a2fae1c...,b07442b2-3166-4233-b884-90f6803d1bfd5d0a2fae1c...,b07442b2-3166-4233-b884-90f6803d1bfd5d0a2fae1c...,None,MALE,PHH,PRIME,DAILY,DAILY,MEDIUM_INCOME,ONLY_LINK,NON_GC
3,20250115,Exp1_MaxBanner_Zepto,PostOrderArrived3,Delhi,android,656adcfd044b093882cd1ff5,Bike Lite,arrived,849da0b9-fd51-4047-9d7d-6fcd709b820e656adcfd04...,849da0b9-fd51-4047-9d7d-6fcd709b820e656adcfd04...,849da0b9-fd51-4047-9d7d-6fcd709b820e656adcfd04...,None,FEMALE,PHH,ELITE,DAILY,DAILY,MEDIUM_INCOME,ONLY_LINK,UNKNOWN
4,20250115,Exp1_MaxBanner_Zepto,PostOrderOnTheWay3,Chennai,android,6439178ecc7cb24c047ba197,Auto,onTheWay,a516d02e-d1f5-40e5-b957-a2e7251be96c6439178ecc...,a516d02e-d1f5-40e5-b957-a2e7251be96c6439178ecc...,a516d02e-d1f5-40e5-b957-a2e7251be96c6439178ecc...,None,FEMALE,PHH,GOLD,BI_WEEKLY,MONTHLY,LOW_INCOME,UNKNOWN,UNKNOWN


#### Bivariate data 

In [25]:
display(get_funnel(df, ['os']))

,format,os,responses,% render rate,% view,% ctr
0,maxBanner,android,1552970,75.79,76.89,0.25


In [23]:
display(get_funnel(df, ['yyyymmdd']))

,format,yyyymmdd,responses,% render rate,% view,% ctr
4,maxBanner,20250119,29,96.55,82.14,4.35
5,maxBanner,20250120,278302,84.01,83.41,0.75
0,maxBanner,20250115,219413,81.52,79.73,0.17
6,maxBanner,20250121,281593,72.69,74.86,0.10
7,maxBanner,20250122,288297,72.64,75.25,0.10
2,maxBanner,20250117,252570,72.17,73.32,0.09
1,maxBanner,20250116,232757,72.12,73.21,0.08
3,maxBanner,20250118,9,100.00,88.89,0.00


In [24]:
display(get_funnel(df, ['slotUnitId']))

,format,slotUnitId,responses,% render rate,% view,% ctr
2,maxBanner,PostOrderStarted3,388903,79.32,81.08,0.42
1,maxBanner,PostOrderOnTheWay3,799900,72.43,70.46,0.21
0,maxBanner,PostOrderArrived3,364167,79.40,85.32,0.14


In [26]:
display(get_funnel(df, ['service']))

,format,service,responses,% render rate,% view,% ctr
4,maxBanner,Auto Pool,302,81.13,75.92,1.08
3,maxBanner,Auto Pet,389,46.79,73.08,0.75
7,maxBanner,Bike Pink,213,86.38,85.33,0.64
10,maxBanner,CabPremium,29739,74.11,68.93,0.64
11,maxBanner,E rickshaw,2237,82.12,79.70,0.48
5,maxBanner,Bike Lite,102943,83.13,80.61,0.38
8,maxBanner,C2C,12239,79.91,82.68,0.36
0,maxBanner,Auto,570793,74.49,77.77,0.28
2,maxBanner,Auto NCR,2265,81.02,78.64,0.28
6,maxBanner,Bike Metro,28306,80.82,79.62,0.24


In [27]:
display(get_funnel(df, ['gender']))

,format,gender,responses,% render rate,% view,% ctr
3,maxBanner,UNKNOWN,117418,78.34,74.87,0.35
2,maxBanner,OTHERS,6570,79.22,72.70,0.34
0,maxBanner,FEMALE,427815,76.32,78.03,0.29
1,maxBanner,MALE,1001167,75.24,76.68,0.22


In [28]:
display(get_funnel(df, ['ltr_segment']))

,format,ltr_segment,responses,% render rate,% view,% ctr
1,maxBanner,NEW,84914,79.42,73.89,0.40
3,maxBanner,UNKNOWN,21365,78.46,68.10,0.35
0,maxBanner,HH,152742,77.67,75.00,0.30
2,maxBanner,PHH,1293949,75.28,77.48,0.23


In [29]:
display(get_funnel(df, ['retention_segments']))

,format,retention_segments,responses,% render rate,% view,% ctr
8,maxBanner,UNKNOWN,106279,79.23,72.74,0.39
4,maxBanner,INACTIVE,88779,77.29,76.08,0.32
0,maxBanner,DORMANT,209425,75.76,76.79,0.29
3,maxBanner,HH,71293,77.63,74.91,0.25
5,maxBanner,PLATINUM,298541,75.35,77.62,0.24
2,maxBanner,GOLD,217031,75.69,77.23,0.23
7,maxBanner,SILVER,105899,76.05,76.77,0.22
1,maxBanner,ELITE,439886,74.70,77.82,0.21
6,maxBanner,PRIME,15837,74.45,78.44,0.17


In [30]:
display(get_funnel(df, ['regularity_segment']))

,format,regularity_segment,responses,% render rate,% view,% ctr
4,maxBanner,UNKNOWN,259021,78.31,74.06,0.34
2,maxBanner,MONTHLY,140102,75.93,77.45,0.26
3,maxBanner,RARE_NEED,140347,76.13,77.37,0.26
0,maxBanner,BI_WEEKLY,273190,75.75,77.21,0.24
5,maxBanner,WEEKLY,374110,75.05,77.76,0.24
1,maxBanner,DAILY,366200,74.60,77.46,0.20


In [31]:
display(get_funnel(df, ['fe_regularity_segment']))

,format,fe_regularity_segment,responses,% render rate,% view,% ctr
4,maxBanner,UNKNOWN,275021,77.61,75.72,0.33
2,maxBanner,MONTHLY,200163,76.28,76.69,0.26
0,maxBanner,BI_WEEKLY,357895,75.63,77.20,0.24
5,maxBanner,WEEKLY,247981,75.20,77.61,0.24
3,maxBanner,RARE_NEED,74406,75.93,75.66,0.23
1,maxBanner,DAILY,397504,74.75,77.35,0.20


In [32]:
display(get_funnel(df, ['income_segment']))

,format,income_segment,responses,% render rate,% view,% ctr
3,maxBanner,UNKNOWN,360197,75.58,75.79,0.30
1,maxBanner,LOW_INCOME,81883,81.89,78.13,0.28
2,maxBanner,MEDIUM_INCOME,496059,80.77,78.26,0.26
0,maxBanner,HIGH_INCOME,614831,71.08,76.14,0.20


In [33]:
display(get_funnel(df, ['service_affinity']))

,format,service_affinity,responses,% render rate,% view,% ctr
7,maxBanner,UNKNOWN,705023,76.66,76.23,0.28
4,maxBanner,ONLY_AUTO,285642,72.65,78.44,0.27
0,maxBanner,AUTO_CAB,22686,72.67,76.07,0.26
1,maxBanner,LINK_AUTO,44260,74.67,78.40,0.22
3,maxBanner,NO_AFFINITY,104799,72.96,76.42,0.21
6,maxBanner,ONLY_LINK,319264,78.15,77.76,0.20
5,maxBanner,ONLY_CAB,62700,74.46,73.50,0.19
2,maxBanner,LINK_CAB,8596,79.11,74.09,0.14


In [34]:
display(get_funnel(df, ['gc_tag']))

,format,gc_tag,responses,% render rate,% view,% ctr
2,maxBanner,UNKNOWN,1238371,75.95,76.75,0.26
1,maxBanner,NON_GC,283779,75.02,77.35,0.20
0,maxBanner,GC,30820,76.21,78.60,0.17
